In [ ]:
import requests
from bs4 import BeautifulSoup
import time
import re
from pyspark.sql import SparkSession, Row

spark = SparkSession.builder.appName("ParuVenduScraper").getOrCreate()

def scraper_paruvendu(nb_pages=5):

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
        "Accept-Language": "fr-FR,fr;q=0.9",
    }

    collecte = []

    for page in range(1, nb_pages + 1):
        print(f"=== Page {page}/{nb_pages} ===")

        # ✅ URL correcte avec pagination
        if page == 1:
            url = "https://www.paruvendu.fr/immobilier/annonceimmofo/liste/?tt=1&vl=Lyon"
        else:
            url = f"https://www.paruvendu.fr/immobilier/annonceimmofo/liste/?tt=1&vl=Lyon&p={page}"

        try:
            response = requests.get(url, headers=headers, timeout=15)

            if response.status_code == 200:
                soup = BeautifulSoup(response.content, "html.parser")
                annonces = soup.find_all("div", class_="blocAnnonce")
                print(f"{len(annonces)} annonces trouvées")

                for annonce in annonces:
                    try:
                        
                        texte = annonce.get_text(separator=" | ", strip=True)
            
                        annonce_id = annonce.get("data-id", None)

                    
                        lien = annonce.find("a", href=True)
                        titre = lien.get("title", "").strip() if lien else None

                
                        prix_match = re.search(r'([\d\s]+€)', texte)
                        prix = prix_match.group(1).strip() if prix_match else None

                       
                        surface_match = re.search(r'(\d+)\s*m[\s\|²]', texte)
                        surface = surface_match.group(1) + " m²" if surface_match else None

                        
                        pieces_match = re.search(r'(\d+)\s*pi', texte, re.IGNORECASE)
                        pieces = pieces_match.group(1) if pieces_match else None

                    
                        chambres_match = re.search(r'(\d+)\s*chambre', texte, re.IGNORECASE)
                        chambres = chambres_match.group(1) if chambres_match else None

            
                        ville_match = re.search(r'([A-ZÀ-Ÿa-zà-ÿ\-\s]+)\s*\((\d{2})\)', texte)
                        localisation = ville_match.group(0).strip() if ville_match else "Lyon"

                       
                        desc_match = re.search(r'([\w\s,àéèêëùûüîïôç\.\-]{80,})', texte)
                        description = desc_match.group(1).strip() if desc_match else None

                    
                        date_match = re.search(r'(\d{2}/\d{2}/\d{4})', texte)
                        date = date_match.group(1) if date_match else None

                        collecte.append(Row(
                            id=annonce_id,
                            titre=titre,
                            prix=prix,
                            surface=surface,
                            pieces=pieces,
                            chambres=chambres,
                            localisation=localisation,
                            description=description,
                            date=date,
                            source="paruvendu"
                        ))

                    except Exception as e:
                        print(f"Erreur annonce : {e}")
                        continue

            else:
                print(f"Erreur HTTP : {response.status_code}")

        except Exception as e:
            print(f"Erreur page {page} : {e}")

        time.sleep(2)

    return collecte

#
donnees = scraper_paruvendu(nb_pages=5)

if donnees:
    df = spark.createDataFrame(donnees)
    df.show(5, truncate=False)
    print(f"\nTotal : {df.count()} annonces")
    df.printSchema()
else:
    print("Aucune donnée collectée")

=== Page 1/5 ===
30 annonces trouvées
=== Page 2/5 ===
30 annonces trouvées
=== Page 3/5 ===
30 annonces trouvées
=== Page 4/5 ===
30 annonces trouvées
=== Page 5/5 ===
30 annonces trouvées
+----------+--------------------------------+---------+-------+------+--------+----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------+---------+
|id        |titre                           |prix     |surface|pieces|chambres|localisation    |description                                                                                                                                                                                                                                                                        |date      |source   |
+----------+----------